In [ ]:
from pathlib import Path
import json, subprocess, sys, zipfile, shutil, time
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
m=list(INPUT.rglob('active_tb_safe_full_v1/SAFE_BOUNDARY.json'))
if len(m)==1: SRC=m[0].parent
else:
 a=list(INPUT.rglob('active_tb_safe_full_v1.zip')); assert len(a)==1,a
 d=WORK/'att'
 with zipfile.ZipFile(a[0]) as zf: zf.extractall(d)
 SRC=d/'active_tb_safe_full_v1'
print(json.loads((SRC/'SAFE_BOUNDARY.json').read_text()))
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','yosys','iverilog','z3'],check=True)
print('tools ready', flush=True)


In [ ]:
# Stage 1: emit both channels for all 156 tasks, both frontends.
for fe in ('correctbench','autobench'):
    OUT=WORK/f'safe_{fe}'; OUT.mkdir(exist_ok=True)
    t0=time.time(); print('='*28, fe, '='*28, flush=True)
    r=subprocess.run([sys.executable,str(SRC/'safe'/'run_safe.py'),
      '--frontend',fe,'--tasks',str(SRC/'amp'/'all_tasks.json'),
      '--bounds','4,8,16,32','--timeout','60','--max-pairs','4',
      '--out',str(OUT)],text=True,capture_output=True,cwd=SRC)
    print(r.stdout[-4000:])
    if r.returncode: print('STDERR:',r.stderr[-3000:])
    print(f'{fe} generation {time.time()-t0:.0f}s', flush=True)


In [ ]:
# Stage 2: sealed evaluation against Golden and Hidden Bank A.
EV=WORK/'eval_full'; EV.mkdir(exist_ok=True)
for fe in ('correctbench','autobench'):
    OUT=WORK/f'safe_{fe}'
    if not (OUT/'safe_manifest.json').is_file(): continue
    t0=time.time(); print('='*28, 'eval', fe, '='*28, flush=True)
    r=subprocess.run([sys.executable,str(SRC/'safe'/'sealed_dual_eval.py'),
      '--suites',str(OUT/'suites'),'--manifest',str(OUT/'safe_manifest.json'),
      '--checkers',str(SRC/'baselines/ConfiBench/data'/('CorrectBenchResults' if fe=='correctbench' else 'AutoBenchResults')/'run_info_and_codes.jsonl'),
      '--golden',str(SRC/'golden.jsonl'),'--mutants',str(SRC/'hidden_bank_a.jsonl'),
      '--out',str(EV/f'{fe}_records.csv')],text=True,capture_output=True)
    print(r.stdout[-2500:])
    if r.returncode: print('STDERR:',r.stderr[-2500:])
    print(f'{fe} eval {time.time()-t0:.0f}s', flush=True)
    shutil.copy2(OUT/'safe_manifest.json', EV/f'{fe}_manifest.json')
shutil.make_archive(str(WORK/'eval_full'),'zip',EV)
print('packaged')


In [ ]:
# Stage 3: publication-compatible comparison against each frontend.
CMP=WORK/'sota_cmp'; CMP.mkdir(exist_ok=True)
for fe,dirn in (('correctbench','CorrectBenchResults'),('autobench','AutoBenchResults')):
    OUT=WORK/f'safe_{fe}'
    if not (OUT/'safe_manifest.json').is_file(): continue
    t0=time.time(); print('='*24,'sota',fe,'='*24, flush=True)
    r=subprocess.run([sys.executable,str(SRC/'safe'/'sota_compare.py'),
      '--suites',str(OUT/'suites'),'--manifest',str(OUT/'safe_manifest.json'),
      '--frontend-records',str(SRC/'baselines/ConfiBench/data'/dirn/'run_info_and_codes.jsonl'),
      '--golden',str(SRC/'golden.jsonl'),'--mutants',str(SRC/'hidden_bank_a.jsonl'),
      '--out',str(CMP/f'{fe}_sota.csv')],text=True,capture_output=True)
    print(r.stdout[-3000:])
    if r.returncode: print('STDERR:',r.stderr[-2000:])
    print(f'{fe} sota {time.time()-t0:.0f}s', flush=True)
shutil.make_archive(str(WORK/'sota_cmp'),'zip',CMP)
print('packaged sota')
